In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, DownloadMode
from pathlib import Path

import os
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_ENABLE_SYMLINKS"] = "1"

pd.set_option('display.max_colwidth', None)

### 1. Загрузка датасета

In [ ]:
# Вариант 1: скачивания через datasets - отключен из-за конфликтов на ОС Windows

# print("Loading datasets...")
# files = {
#     "jailbreak": "hf://datasets/youbin2014/JailbreakDB/text_jailbreak_unique.csv",
#     "regular":   "hf://datasets/youbin2014/JailbreakDB/text_regular_unique.csv",
# }
#
# raw_dataset = load_dataset("csv", data_files=files, token=hf_token)
#
# print("Datasets loaded successfully")



In [ ]:
# Вариант 2: файлы скачаны вручную в task-4/datasets/
# Файл 1: https://huggingface.co/datasets/youbin2014/JailbreakDB/resolve/main/text_jailbreak_unique.csv
# Файл 2: https://huggingface.co/datasets/youbin2014/JailbreakDB/resolve/main/text_regular_unique.csv



In [ ]:
print("Loading datasets from local disk...")

dataset_dir = Path(".") / "datasets"

path_dataset_attack = dataset_dir / "text_jailbreak_unique.csv"
path_dataset_normal = dataset_dir / "text_regular_unique.csv"

# Проверяем наличие файлов и читаем
if not path_dataset_attack.exists() or not path_dataset_normal.exists():
    raise FileNotFoundError(
        f"Missing files. Check files in directory: {dataset_dir.absolute()}\n"
        "Two files expected:\n"
        "- text_jailbreak_unique.csv\n"
        "- text_regular_unique.csv"
    )

df_attack = pd.read_csv(
    path_dataset_attack,
    engine="python",
    on_bad_lines='skip'
)

df_normal = pd.read_csv(
    path_dataset_normal,
    engine="python",
    on_bad_lines='skip'
)

print("Datasets loaded successfully")


In [ ]:
print(f"Датасет атак:\n"
      f"- количество строк: {df_attack.shape[0]}\n"
      f"- колонки: {list(df_attack.columns)}\n")
print(f"Датасет нейтральных промтов:\n"
      f"- количество строк: {df_normal.shape[0]}\n"
      f"- колонки: {list(df_attack.columns)}\n")


### 2. Склеивание датасетов и анализ базовых свойств

In [ ]:
df_combined = pd.concat([df_attack, df_normal])

#### Проверка пустых значений

In [ ]:
df_combined.isna().mean()

##### Выводы:
- Пустые значения есть только в колонке system_prompt.
- Порядка 3/4 примеров в датасете не имеют системных промтов.

#### Проверка баланса классов атака / не атака

In [ ]:
df_combined['jailbreak'].mean()

##### Выводы:
- доля примеров атак в датасете ~29%
- сильного перекоса нет

### 3. Проверка шума и пересечений

#### Поиск пересечений (одинаковый пример промта присутствует как в примерах атак, так и в примерах не атак)

In [ ]:
overlap = set(df_attack['user_prompt']).intersection(set(df_normal['user_prompt']))

In [ ]:
len(overlap)

In [ ]:
print(f"Найдено пересечений: {len(overlap)}")

##### Выводы:
- 8 примеров промтов помечены одновременно как атаки, и как не атаки.
- Необходимо полностью устранить эти промты, так как нам не известно достоверно к какому классу они относятся.

#### Очистка пересечений

In [ ]:
df_combined = df_combined[~df_combined['user_prompt'].isin(overlap)]

In [ ]:
print(f"Количество строк после очистки пересечений: {df_combined.shape[0]}")

#### Очистка дубликатов
Удаляем строки с дубликатами промтов - после очистки пересечений между классами атака / не атака это делать безопасно.

In [ ]:
df_combined = df_combined.drop_duplicates(subset=['user_prompt'], keep='first', ignore_index=True)

#### Анализ шума по колонке tactic

In [ ]:
df_attack['tactic'].value_counts()

In [ ]:
df_normal['tactic'].value_counts()

In [ ]:
df_attack[df_attack['tactic']==0].head()

In [ ]:
df_attack[df_attack['tactic']==1].head()

In [ ]:
df_normal[df_normal['tactic']==0].head()

In [ ]:
df_normal[df_normal['tactic']==1].head()

##### Выводы:
- В описании датасета колонка tactic описана так: "Prompt tactic or category when available."
- Это довольно размытое описание и трудно заключить что такое тактика 0 или тактика 1, возможно имеются ввиду более простые или более сложные промты, либо более и менее очевидно подходящие к классу атака / не атака.
- В датасете атак ~85% строк имеют tactic = 1, остальные - tactic = 0.
- В датасете не атак - наоборот: ~91% строк имеют tactic = 0, остальные - tactic = 1.
- Беглый осмотр показывает, что tactic = 0 для атак и tactic = 1 для не атак - не являются мусором.
- Удалять эти строки не следует.

### 4. Текстовый анализ признаков

#### Анализ длин промтов

In [ ]:
df_nlp = df_combined.copy()
df_nlp['user_prompt'] = df_nlp['user_prompt'].astype(str)

df_nlp['char_len'] = df_nlp['user_prompt'].apply(len)
df_nlp['word_len'] = df_nlp['user_prompt'].apply(lambda x: len(x.split()))

metrics = df_nlp.groupby('jailbreak')[['char_len', 'word_len']].agg(['mean', 'median', 'max'])
print("Статистика длин промтов:")
display(metrics)

In [ ]:
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

sns.kdeplot(
    data=df_nlp[df_nlp['jailbreak'] == 0],
    x='word_len',
    label='Нейтральные промты (0)',
    fill=True,
    color='#2ecc71',
    alpha=0.4
)
sns.kdeplot(
    data=df_nlp[df_nlp['jailbreak'] == 1],
    x='word_len',
    label='Промт-атаки (1)',
    fill=True,
    color='#e74c3d',
    alpha=0.4
)

plt.xlim(0, 500)

plt.title("Распределение длин промтов в словах (срез до 500 слов)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Количество слов в промте", fontsize=12)
plt.ylabel("Плотность распределения", fontsize=12)
plt.legend()

plt.show()

##### Выводы:
- Между датасетами атак и не атак - есть существенная разница в параметрах текстов промтов: среди не атак преобладают более короткие тексты (до 100 слов в промте), среди атак заметно больше более длинных текстов.
    - Есть риск что модель будет выучит ложную корреляцию между длиной текста и признаком атаки.
    - Необходимо выбирать модель, на которую в меньшей степени повлияет длина текста, например, Modern BERT.
    - Необходимо учесть риск некорректного обучения и подготовить тестовую выборку, например с длинными текстами не атак или наоборот - чтобы удостовериться что модель не запомнила лишнюю коррелцию.
- Судя по анализу длины текстов - 99% примеров укладываются в 1024 токена; необходимо учесть это чтобы оптимизировать обучение.

#### Частотный анализ слов (N-grams)

In [ ]:
def get_top_ngrams(corpus, n=2, top_k=20):
    vec = CountVectorizer(ngram_range=(n, n), stop_words='english') .fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return pd.DataFrame(words_freq[:top_k], columns=['Фраза', 'Количество'])

attacks_text = df_nlp[df_nlp['jailbreak'] == 1]['user_prompt'].dropna()
normal_text = df_nlp[df_nlp['jailbreak'] == 0]['user_prompt'].dropna()

print("Топ 20 сигнатур в промт-атаках (биграммы):")
display(get_top_ngrams(attacks_text, n=2, top_k=20))

print("Топ 20 фраз в нейтральных промтах (биграммы):")
display(get_top_ngrams(normal_text, n=2, top_k=20))


##### Выводы
- Атаки нельзя выделить по ключевым словам: топ 20 сочетаний в атаках выглядят вполне нейтрально.
- Необходимо использовать модель, которая будет учитывать контекст, а не просто частоту появления слов, например, Modern BERT.

# Общие выводы по итогам EDA

1. Данные в датасетах относительно чистые, без явных перекосов по классам атака / не атака.
2. Требуется подготовка датасета:
    - Слить два датасета в один.
    - Очистить от пересечений (один и тот же промт присутствует с признаком атака и не атака).
    - Удалить дубли (по полю user_prompt).
    - Удалить колонки system_prompt, source, tactic.
3. Необходимо использовать модель, учитывающую контекст, например, Modern BERT.

# Стратегия формирования выборки для обучения модели и валидации

### Формирование выборки

### Шаг 1: первичная подготовка данных
    - Слить два датасета в один.
    - Очистить от пересечений (один и тот же промт присутствует с признаком атака и не атака).
    - Удалить дубли (по полю user_prompt).
    - Удалить колонки system_prompt, source, tactic.

### Шаг 2: выборка для стресс-теста

- Изымаем из общей выборки длинные тексты промтов, которые не являются атакой.
- На этих примерах не будем обучать модель.
- На этих примерах будем тестировать обученную модель.
- Таким образом, провалидируем, что модель не запомнила ложную корреляцию между длиной промта и признаком атаки.

### Шаг 3: стратифицированный сплит

Делим общую выборку на три части в пропорции 80 / 10 / 10:
- Выборка Train - 80%
- Выборка Validation - 10%
- Выборка Test - 10%

### Стратегия валидации

### Проверка основных метрик:
- Recall
- Precision
- F1-Score

### Валидация отсутствия перекоса на основе длины промтов:
- Выполнить отдельный тест на специальной выборке длинных промтов, которые являются не атаками.

### Сэмплирование:
Для обучения модели на слабом оборудовании может потребоваться взять сэмпл данных вместо всего датасета. В этом случае необходимо учитывать:
- Целевой класс: пропорция признаков атака / не атака в сэмпле - должна быть такой же, как и в основном датасете.
- Бакеты длины: разделим все промты на 4 группы по длине текста: короткие, средние, длинные и сверх-длинные. Создадим признак класс (атака / не атака) + бакет длины, например: 0_long, 1_short, 1 super-long.